# 🔗 STACKING MULTIMODAL — **v1 BASE** (meta-modelo LR/XGB + interpretación)
## TFM · Fusión de las tres modalidades · Universidad de Salamanca

---

Fusión **tardía** (*late fusion*) de radiografía, ECG y analíticas: cada modalidad ya entrenó por su
cuenta y exportó sus probabilidades; aquí un **meta-modelo por etiqueta** aprende a combinarlas.
Entrada del meta: **18 probabilidades** (3 modalidades × 6 etiquetas) **+ contexto clínico**.

## ⚠️ Requisito de ejecución
Necesita los CSV `*_pred_{oof_train,val,test}.csv` de las **tres** modalidades. Va **el último**.
Rutas verificadas: `salidas/01_cxr/v2`, `salidas/02_ecg/v2`, `salidas/03_labs/v2`.

## 🔄 Adaptación a las conclusiones del EDA

| Cambio | Detalle | Origen |
|---|---|---|
| **Etiquetado FINAL** | `POS=(==1)` · `NEG=(==0)|(NaN→0)` · **−1 ENMASCARADO**. | §3 |
| **Métrica primaria = AUC-PR** | `macro_AP_path` para elegir el meta-modelo y tunear `C`; ROC secundaria; **IC bootstrap**. | §3 · §1 |
| **Fusión vs mejor mono** | Se comparan las 3 fusiones (LR, XGB, promedio) contra las **3 modalidades solas**, y se declara si los **IC se solapan**. | §11 · §5 |
| **Recalibración post-fusión** | Isotónica reajustada tras fusionar + Brier antes/después. | §11 |
| **Puntos de operación** | F1, cribado (Se≥0,90) y confirmación (Sp≥0,90) con Se/Sp/VPP/VPN. | §11 |
| **Equidad y robustez** | Estratificación por sexo, etnia, ingreso y proyección AP/PA. | §2/§9 |

### ⚠️ Corrección importante del etiquetado
Este notebook usaba `uncertainty_policy="zeros"` (el **−1 contaba como negativo** y **no** se
enmascaraba) y `derive=True`. Era **incoherente con las tres modalidades**, que sí enmascaran el −1:
el meta-modelo estaba aprendiendo a combinar probabilidades entrenadas contra un objetivo distinto
del que luego se evaluaba. Corregido a `"ignore"` + `derive=False`.

> Consecuencia: **las cifras de fusión anteriores no son comparables**. Hay que reejecutar.

### Por qué la recalibración no es opcional
Cada modalidad venía calibrada por separado, pero **combinar probabilidades no conserva la
calibración**: el promedio de probabilidades calibradas queda sistemáticamente *sub-confiado*
(comprimido hacia el centro). Por eso se reajusta una isotónica sobre VAL **después** de fusionar y
se vuelve a medir el Brier.

- **`cxr_view` NUNCA como predictor**: no entra en el contexto del meta; solo estratifica.



In [ ]:
# CELDA 1 · DEPENDENCIAS (v2 solo carga CSV + meta-modelos ligeros: no necesita torch/lightgbm)
import subprocess, sys
for pkg in ["xgboost","scikit-learn","pandas","numpy","matplotlib","seaborn"]:
    subprocess.run([sys.executable,"-m","pip","install",pkg,"-q"],check=True)
print("Dependencias instaladas.")

In [ ]:
# CELDA 2 · IMPORTS / SEMILLAS / CONSTANTES
import os, gc, json, time, copy, random, warnings
from pathlib import Path
from sklearn.isotonic import IsotonicRegression
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve
import xgboost as xgb
warnings.filterwarnings("ignore")
SEED=42; random.seed(SEED); np.random.seed(SEED); DEVICE="cpu"

NB=Path.cwd()
BASE=Path(r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0")
CSV_DIR=BASE/"data_csv"/"clean"
TRAIN_CSV,VAL_CSV,TEST_CSV=CSV_DIR/"train_clean.csv",CSV_DIR/"val_clean.csv",CSV_DIR/"test_clean.csv"
# v2: las 3 modalidades exportan ya su OOF/val/test -> aquí SOLO se cargan los CSV (ver notebooks 01/02/03)
CXR_OOF =Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\01_cxr\v2")     # CXR  v5   -> cxr_pred_*.csv
ECG_OOF =Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\02_ecg\v2")      # ECG  v3.1 -> ecg_pred_*.csv
LABS_OOF=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\03_labs\v2")      # LABS v2.2 -> labs_pred_*.csv
OUT=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\04_stacking\v1"); OUT.mkdir(parents=True,exist_ok=True)
FIG=OUT/"figuras"; FIG.mkdir(parents=True,exist_ok=True)

LABELS=["Atelectasis","Cardiomegaly","Edema","Lung Opacity","No Finding","Pleural Effusion"]
N_LABELS=len(LABELS); NO_FINDING="No Finding"
PATHOLOGY_LABELS=[l for l in LABELS if l!=NO_FINDING]; CORE_LABELS=["Cardiomegaly","Edema","Pleural Effusion"]
MODALITIES=["CXR","ECG","LABS"]
GENDER_MAP={0:0,1:1,"0":0,"1":1,"M":1,"F":0}
RACE_MAP={"UNKNOWN":0,"WHITE":1,"BLACK":2,"ASIAN":3,"HISPANIC_LATINO":4,"OTHER_KNOWN":0}
ADMISSION_MAP={"SCHEDULED":0,"EMERGENCY":1,"OBSERVATION":2,"URGENT":3}
ADM_LOC_MAP={"EMERGENCY_ROOM":0,"REFERRAL":1,"TRANSFER":2,"INTRA_HOSPITAL":3}; RACE_ONEHOT=5
print(f"Stacking v2 (solo carga OOF) · {DEVICE}")
print(f"dirs OOF -> CXR:{CXR_OOF.exists()} ECG:{ECG_OOF.exists()} LABS:{LABS_OOF.exists()}")


In [ ]:
# CELDA 3 · CARGA, OBJETIVOS (masking + negativos derivados) Y CONTEXTO
df_train=pd.read_csv(TRAIN_CSV,sep=";"); df_val=pd.read_csv(VAL_CSV,sep=";"); df_test=pd.read_csv(TEST_CSV,sep=";")
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · build_targets
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : convierte los estados 1/0/−1/NaN de cada etiqueta en (labels, mask).
# POR QUÉ    : el meta-modelo DEBE usar EXACTAMENTE la misma definición de etiqueta que las tres
#              modalidades; si no, estaría aprendiendo a combinar probabilidades entrenadas contra
#              un objetivo distinto del que se evalúa, y la comparación fusión-vs-mono no valdría.
#              POS = (==1) · NEG = (==0) | (NaN→0) · −1 = ENMASCARADO (U-Ignore).
# ENTRADAS   : df (split) · uncertainty_policy ("ignore") · derive (compatibilidad) · verbose
# SALIDAS    : (labels (N,6) float32, mask (N,6) float32)
# ORIGEN EDA : §3 · "negativo = 0 explícito + NaN→0; el −1 se enmascara; «Sin hallazgo» NO como negativo".
# ⚠️ CAMBIO IMPORTANTE vs la versión previa de ESTE notebook: usaba `uncertainty_policy="zeros"`
#    (el −1 contaba como negativo y NO se enmascaraba) y `derive=True`. Era INCOHERENTE con las tres
#    modalidades, que sí enmascaran el −1. Corregido a "ignore" + derive=False.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#    verificar pos_weight ≈ 2,28/1,85/3,43/2,15/6,31/1,66; y avisar de que las cifras de fusión
#    anteriores NO son comparables (se evaluaban contra otro objetivo).
# ══════════════════════════════════════════════════════════════════════════════
def build_targets(df, uncertainty_policy="ignore", derive=False, verbose=False):
    raw=df[LABELS].to_numpy(dtype=float); N=raw.shape[0]
    labels=(raw==1.0).astype(np.float32)          # 1 → positivo ; 0 y NaN → 0 (negativo)
    mask  =np.ones((N,N_LABELS),np.float32)       # por defecto TODO entra en pérdida/métrica
    unc=(raw==-1.0)
    if   uncertainty_policy=="ignore": mask[unc]=0.0    # −1 → ENMASCARADO (definición FINAL)
    elif uncertainty_policy=="ones":   labels[unc]=1.0
    if derive:   # CONSERVADO por compatibilidad; NO forma parte de la definición FINAL
        nf=LABELS.index(NO_FINDING); pc=[j for j in range(N_LABELS) if j!=nf]; nfp=(raw[:,nf]==1)
        for j in pc:
            f=nfp&np.isnan(raw[:,j]); labels[f,j]=0.0
        ap=(raw[:,pc]==1).any(1); fn=ap&np.isnan(raw[:,nf]); labels[fn,nf]=0.0
    if verbose:
        for j,l in enumerate(LABELS):
            s=mask[:,j]==1; p=int((labels[s,j]==1).sum()); n=int((labels[s,j]==0).sum())
            print(f"   {l:18s} pos={p:5d} neg={n:5d} enmasc={int((mask[:,j]==0).sum()):4d} pos_weight={n/max(p,1):.2f}")
    return labels,mask
y_train,m_train=build_targets(df_train,verbose=True); y_val,m_val=build_targets(df_val); y_test,m_test=build_targets(df_test)

def build_context(df):
    N=len(df); cols=[]; names=[]
    AGEMIN,AGEMAX=float(df_train["age"].min()),float(df_train["age"].max())
    HMIN,HMAX=float(df_train["hours_adm_to_cxr"].min()),float(df_train["hours_adm_to_cxr"].max())
    cols.append(((df["age"].astype(float)-AGEMIN)/(AGEMAX-AGEMIN+1e-8)).to_numpy()[:,None]); names.append("Edad")
    cols.append(df["gender"].map(lambda v:float(GENDER_MAP.get(v,0))).to_numpy()[:,None]); names.append("Sexo")
    def oh(series,mp,pref):
        M=np.zeros((N,max(mp.values())+1),np.float32)
        for i,v in enumerate(series): M[i,mp.get(str(v).upper(),0)]=1.0
        return M,[f"{pref}={k}" for k,_ in sorted(mp.items(),key=lambda x:x[1])][:M.shape[1]]
    for col,mp,pref in [("race",RACE_MAP,"Raza"),("admission_type",ADMISSION_MAP,"Ingreso"),("admission_location",ADM_LOC_MAP,"Lugar")]:
        M,nm=oh(df[col],mp,pref); cols.append(M); names+=nm
    cols.append(((df["hours_adm_to_cxr"].astype(float)-HMIN)/(HMAX-HMIN+1e-8)).to_numpy()[:,None]); names.append("Horas")
    return np.hstack(cols).astype(np.float32),names
CTX_tr,CTX_NAMES=build_context(df_train); CTX_vl,_=build_context(df_val); CTX_te,_=build_context(df_test)
print(f"train={len(df_train)} val={len(df_val)} test={len(df_test)} · contexto={CTX_tr.shape[1]} dims")



In [ ]:
# CELDA 4 · MÉTRICAS
def multilabel_metrics(probs,labels,mask,thresholds=None):
    if thresholds is None: thresholds={l:0.5 for l in LABELS}
    res={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]; npos=int(yt.sum()); nneg=int((1-yt).sum())
        thr=thresholds.get(l,0.5); pred=(yp>=thr).astype(float)
        if npos>=2 and nneg>=2: auc=roc_auc_score(yt,yp); ap=average_precision_score(yt,yp)
        else: auc=ap=float("nan")
        tp=int(((pred==1)&(yt==1)).sum()); tn=int(((pred==0)&(yt==0)).sum()); fp=int(((pred==1)&(yt==0)).sum()); fn=int(((pred==0)&(yt==1)).sum())
        res[l]={"AUC":auc,"AP":ap,"F1":f1_score(yt,pred,zero_division=0),"sens":tp/max(tp+fn,1),"spec":tn/max(tn+fp,1),"prevalencia":npos/max(npos+nneg,1),"n_pos":npos,"n_neg":nneg,"thr":thr,"TP":tp,"TN":tn,"FP":fp,"FN":fn}
    def mac(g,k):
        v=[res[l][k] for l in g if not np.isnan(res[l][k])]; return float(np.mean(v)) if v else float("nan")
    res["macro_AUC_core"]=mac(CORE_LABELS,"AUC"); res["macro_AUC_path"]=mac(PATHOLOGY_LABELS,"AUC")   # secundaria
    res["macro_AP_core"] =mac(CORE_LABELS,"AP");  res["macro_AP_path"] =mac(PATHOLOGY_LABELS,"AP")    # PRIMARIA
    return res
def best_thresholds_by_f1(probs,labels,mask,grid=None):
    if grid is None: grid=np.linspace(0.05,0.95,37)
    thr={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]
        if yt.sum()<2: thr[l]=0.5; continue
        bf,bt=-1,0.5
        for t in grid:
            f=f1_score(yt,(yp>=t).astype(float),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[l]=float(bt)
    return thr
def dyn_pos_weight(y,m,clip=10.0):
    w=np.ones(N_LABELS,np.float32)
    for j in range(N_LABELS):
        s=m[:,j]==1; pos=(y[s,j]==1).sum(); neg=(y[s,j]==0).sum(); w[j]=np.clip(neg/max(pos,1),1/clip,clip)
    return w
print("Métricas listas.")



In [ ]:
# CELDA 4b · KIT DE EVALUACIÓN CLÍNICA (B1, B2, B3, B4, B6, B8) — implementa los recuadros naranjas del EDA que faltaban
# NOTA: B5 (importancia MI/ANOVA) y B7 (dependencia de flags MNAR) NO APLICAN a esta modalidad.
# Motivo (§6 EDA): el ECG es una senal continua SIEMPRE presente: no hay variables tabulares cuya
# importancia medir ni patron de ausencia que vigilar. Esa informacion MNAR entra por el modulo
# tabular y se propaga a la fusion. Aqui si aplican B1, B2, B3, B4, B6 y B8.
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · bootstrap_ci_metric                                          [B4]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : remuestrea con reemplazo y recalcula la métrica, devolviendo el intervalo percentil,
#              POR ETIQUETA y para el MACRO de las 5 patologías.
# POR QUÉ    : con 464 pacientes en test, una diferencia entre modelos puede ser azar; el IC es lo
#              que permite afirmar (o no) que un modelo supera a otro.
# ENTRADAS   : probs (N,6) · labels (N,6) · mask (N,6) · metric "ap"|"auc" · n_boot · alpha
# SALIDAS    : dict {etiqueta:(lo,hi)} + clave "macro_path"
# ORIGEN EDA : §1 · "reportar SIEMPRE IC bootstrap por el tamaño reducido de val/test".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si los IC del stacking y del promedio SE SOLAPAN, no afirmar superioridad del stacking.
# ══════════════════════════════════════════════════════════════════════════════
def bootstrap_ci_metric(probs, labels, mask, metric="ap", n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.RandomState(seed)
    scorer = average_precision_score if metric == "ap" else roc_auc_score
    out, macro_vals = {}, []
    for j, l in enumerate(LABELS):
        idx = np.where(mask[:, j] == 1)[0]; yt, yp = labels[idx, j], probs[idx, j]
        if int(yt.sum()) < 2 or int((1 - yt).sum()) < 2:
            out[l] = (float("nan"), float("nan")); continue
        vals = []
        for _ in range(n_boot):
            bs = rng.randint(0, len(idx), len(idx))
            if yt[bs].sum() < 1 or (1 - yt[bs]).sum() < 1: continue
            vals.append(scorer(yt[bs], yp[bs]))
        out[l] = (float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))) if vals else (float("nan"), float("nan"))
    for _ in range(n_boot):
        bs = rng.randint(0, len(probs), len(probs)); per = []
        for j, l in enumerate(LABELS):
            if l not in PATHOLOGY_LABELS: continue
            sel = mask[bs, j] == 1; yt, yp = labels[bs][sel, j], probs[bs][sel, j]
            if yt.sum() < 1 or (1 - yt).sum() < 1: continue
            per.append(scorer(yt, yp))
        if per: macro_vals.append(np.mean(per))
    out["macro_path"] = (float(np.percentile(macro_vals, 100*alpha/2)),
                         float(np.percentile(macro_vals, 100*(1-alpha/2)))) if macro_vals else (float("nan"), float("nan"))
    return out

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · operating_points                                             [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : fija en VALIDACIÓN tres umbrales por etiqueta: "f1" (equilibrio), "cribado" (el más
#              alto que aún da Se>=sens_target) y "confirm" (el más bajo que aún da Sp>=spec_target).
# POR QUÉ    : un solo umbral no sirve en clínica. Cribar exige no perder enfermos; confirmar exige
#              no alarmar en falso. Son dos decisiones distintas sobre el mismo modelo.
# ENTRADAS   : probs/labels/mask de VALIDACIÓN · sens_target · spec_target
# SALIDAS    : dict {"f1"|"cribado"|"confirm": {etiqueta: umbral}}
# ORIGEN EDA : §11 · "fijar en VAL alta sensibilidad (cribado) y alta especificidad (confirmación).
#              Reportar Se/Sp/VPP/VPN".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              el VPP depende de la PREVALENCIA (13-36 %): un VPP modesto puede valer para cribar y
#              ser inservible para confirmar. Discutir cada punto por su consecuencia clínica.
# ══════════════════════════════════════════════════════════════════════════════
def operating_points(probs, labels, mask, sens_target=0.90, spec_target=0.90):
    grid = np.linspace(0.01, 0.99, 99); pts = {"f1": {}, "cribado": {}, "confirm": {}}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2:
            for k in pts: pts[k][l] = 0.5
            continue
        best_f1, thr_f1 = -1, 0.5; thr_sens, thr_spec = grid[0], grid[-1]
        for t in grid:
            pred = (yp >= t).astype(float)
            tp = ((pred == 1) & (yt == 1)).sum(); fn = ((pred == 0) & (yt == 1)).sum()
            tn = ((pred == 0) & (yt == 0)).sum(); fp = ((pred == 1) & (yt == 0)).sum()
            f1 = f1_score(yt, pred, zero_division=0)
            if f1 > best_f1: best_f1, thr_f1 = f1, t
            if tp/max(tp+fn, 1) >= sens_target: thr_sens = max(thr_sens, t)
            if tn/max(tn+fp, 1) >= spec_target: thr_spec = min(thr_spec, t)
        pts["f1"][l], pts["cribado"][l], pts["confirm"][l] = float(thr_f1), float(thr_sens), float(thr_spec)
    return pts

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · clinical_report                                              [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : evalúa unos umbrales y devuelve Se, Sp, VPP, VPN y la confusión por etiqueta.
# POR QUÉ    : AUC y AP resumen el ranking, pero la decisión se toma en UN umbral; el clínico
#              necesita saber cuántos enfermos se escapan y cuántas alarmas falsas se generan.
# ENTRADAS   : probs/labels/mask (TEST) · thresholds {etiqueta: umbral} · punto (nombre)
# SALIDAS    : DataFrame (punto, etiqueta, umbral, Se, Sp, VPP, VPN, TP/TN/FP/FN, prevalencia)
# ORIGEN EDA : §11 "Reportar Se/Sp/VPP/VPN" · §3 (la prevalencia condiciona el VPP).
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar FALSOS NEGATIVOS en cribado frente a FALSOS POSITIVOS en confirmación.
# ══════════════════════════════════════════════════════════════════════════════
def clinical_report(probs, labels, mask, thresholds, punto="f1"):
    rows = []
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        t = thresholds.get(l, 0.5); pred = (yp >= t).astype(float)
        tp = int(((pred == 1) & (yt == 1)).sum()); tn = int(((pred == 0) & (yt == 0)).sum())
        fp = int(((pred == 1) & (yt == 0)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
        rows.append({"punto": punto, "etiqueta": l, "umbral": round(t, 3),
                     "Se": tp/max(tp+fn,1), "Sp": tn/max(tn+fp,1), "VPP": tp/max(tp+fp,1), "VPN": tn/max(tn+fn,1),
                     "TP": tp, "TN": tn, "FP": fp, "FN": fn, "prevalencia": (tp+fn)/max(tp+tn+fp+fn,1)})
    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · calibration_report                                           [B2]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : Brier score por etiqueta + puntos de la curva de fiabilidad (10 bins por cuantiles).
# POR QUÉ    : la herramienta clínica muestra PROBABILIDADES; si no están calibradas, un 0,8 no
#              significa "80 % de estos pacientes lo tienen" y la cifra engaña al médico.
# ENTRADAS   : probs/labels/mask · n_bins
# SALIDAS    : (DataFrame Brier por etiqueta, dict {etiqueta:(frac_obs, media_pred)})
# ORIGEN EDA : §11 · "verificar con Brier score y curva de fiabilidad; recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar Brier ANTES vs DESPUÉS; si no mejora, decirlo. RECALIBRAR tras la fusión.
# ══════════════════════════════════════════════════════════════════════════════
def calibration_report(probs, labels, mask, n_bins=10):
    rows, curves = [], {}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if len(np.unique(yt)) < 2:
            rows.append({"etiqueta": l, "Brier": float("nan")}); continue
        rows.append({"etiqueta": l, "Brier": float(brier_score_loss(yt, np.clip(yp, 0, 1)))})
        try: curves[l] = calibration_curve(yt, np.clip(yp, 0, 1), n_bins=n_bins, strategy="quantile")
        except Exception: curves[l] = (np.array([]), np.array([]))
    return pd.DataFrame(rows), curves

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · consistency_no_finding                                       [B6]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : correlación entre P("Sin hallazgo") y max P(patología), y % de casos en que ambas
#              superan 0,5 a la vez (el modelo se contradice).
# POR QUÉ    : las 6 cabezas son independientes; nada las obliga a ser coherentes. Un modelo que
#              afirma "sano" y "con derrame" a la vez es inaceptable en una herramienta clínica.
# ENTRADAS   : probs (N,6)
# SALIDAS    : dict {correlación (debe ser NEGATIVA), % incoherentes}
# ORIGEN EDA : §3 · "P(normal) útil como chequeo de consistencia, nunca como fuente de negativos".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              correlación ~0 o positiva ⇒ la cabeza "Sin hallazgo" no aprende normalidad.
# ══════════════════════════════════════════════════════════════════════════════
def consistency_no_finding(probs):
    j_nf = LABELS.index(NO_FINDING); j_p = [j for j in range(N_LABELS) if j != j_nf]
    p_nf, p_max = probs[:, j_nf], probs[:, j_p].max(axis=1)
    return {"corr_NoFinding_vs_maxPatologia": float(np.corrcoef(p_nf, p_max)[0, 1]),
            "pct_incoherentes": float(((p_nf > 0.5) & (p_max > 0.5)).mean()*100)}

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · stratified_report                                            [B8]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : recalcula la métrica primaria dentro de cada subgrupo (sexo, etnia, ingreso) y por
#              PROYECCIÓN radiográfica (cxr_view).
# POR QUÉ    : (a) equidad; (b) robustez — la placa AP se hace al paciente encamado y magnifica la
#              silueta cardíaca, así que conviene ver si el rendimiento depende de la proyección.
# ENTRADAS   : df (metadatos del split) · probs/labels/mask · cols
# SALIDAS    : DataFrame (variable, grupo, n, macro_AP, macro_AUC)
# ORIGEN EDA : §2/§9 "evaluar equidad por sexo y etnia" · ANEXO CXR "monitorizar cxr_view".
# DECISIÓN DE DISEÑO: cxr_view se usa SOLO aquí. NUNCA como predictor: es proxy de gravedad y su
#              inclusión inflaría el resultado por un atajo asistencial en lugar de señal biológica.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              subgrupos con n<60 pueden diferir por PURO RUIDO; no afirmar inequidad sin IC.
# ══════════════════════════════════════════════════════════════════════════════
def stratified_report(df, probs, labels, mask, cols=("gender", "race", "admission_type", "cxr_view")):
    d = df.reset_index(drop=True); rows = []
    for col in cols:
        if col not in d.columns: continue
        for v in sorted(d[col].dropna().unique(), key=str):
            idx = d.index[d[col] == v].to_numpy()
            if len(idx) < 15:
                rows.append({"variable": col, "grupo": str(v), "n": len(idx), "macro_AP": np.nan, "macro_AUC": np.nan}); continue
            mm = multilabel_metrics(probs[idx], labels[idx], mask[idx])
            rows.append({"variable": col, "grupo": str(v), "n": len(idx),
                         "macro_AP": mm["macro_AP_path"], "macro_AUC": mm["macro_AUC_path"]})
    return pd.DataFrame(rows)



In [ ]:
# CELDA 5 · CXR — cargar predicciones OOF/val/test (mejor versión v5) y alinear por hadm_id
def load_cxr(csv):
    d=pd.read_csv(csv).set_index("hadm_id")
    cols=[f"cxr_{l.replace(' ','_')}_cal" for l in LABELS]
    return d[cols]
cxr_oof_df=load_cxr(CXR_OOF/"cxr_pred_oof_train.csv")
cxr_val_df=load_cxr(CXR_OOF/"cxr_pred_val.csv")
cxr_test_df=load_cxr(CXR_OOF/"cxr_pred_test.csv")
def align(df_idx, pred_df):
    out=pred_df.reindex(df_idx["hadm_id"].to_numpy())
    miss=out.isna().any(axis=1).sum()
    if miss>0: print(f"   ⚠ {miss} hadm_id sin predicción CXR (se rellenan con 0.5)"); out=out.fillna(0.5)
    return out.to_numpy(np.float32)
CXR_oof=align(df_train,cxr_oof_df); CXR_vl=align(df_val,cxr_val_df); CXR_te=align(df_test,cxr_test_df)
print(f"CXR cargado · OOF={CXR_oof.shape} val={CXR_vl.shape} test={CXR_te.shape}")
print("   CXR OOF macro (sanity):", round(multilabel_metrics(CXR_oof,y_train,m_train)["macro_AP_path"],4))



In [ ]:
# CELDA 6 · LABS — cargar OOF/val/test exportados por el notebook 03 (blend v2.2). Sin re-entrenar.
def load_mod(csv, prefix):
    d=pd.read_csv(csv).set_index("hadm_id")
    cols=[f"{prefix}_{l.replace(' ','_')}_cal" for l in LABELS]   # probabilidades CALIBRADAS (mejores features de fusión)
    return d[cols]
LABS_oof=align(df_train, load_mod(LABS_OOF/"labs_pred_oof_train.csv","labs"))
LABS_vl =align(df_val,   load_mod(LABS_OOF/"labs_pred_val.csv","labs"))
LABS_te =align(df_test,  load_mod(LABS_OOF/"labs_pred_test.csv","labs"))
print(f"LABS cargado · OOF={LABS_oof.shape} val={LABS_vl.shape} test={LABS_te.shape}")
print("   LABS OOF macro (sanity):", round(multilabel_metrics(LABS_oof,y_train,m_train)["macro_AP_path"],4))


In [ ]:
# CELDA 7 · ECG — cargar OOF/val/test exportados por el notebook 02 (ResNet1D v3.1). Sin re-entrenar.
ECG_oof=align(df_train, load_mod(ECG_OOF/"ecg_pred_oof_train.csv","ecg"))
ECG_vl =align(df_val,   load_mod(ECG_OOF/"ecg_pred_val.csv","ecg"))
ECG_te =align(df_test,  load_mod(ECG_OOF/"ecg_pred_test.csv","ecg"))
print(f"ECG cargado · OOF={ECG_oof.shape} val={ECG_vl.shape} test={ECG_te.shape}")
print("   ECG OOF macro (sanity):", round(multilabel_metrics(ECG_oof,y_train,m_train)["macro_AP_path"],4))


In [ ]:
# CELDA 8 · ENSAMBLAR MATRIZ DEL META-MODELO (18 probas + contexto)
def meta_matrix(cxr,ecg,labs,ctx): return np.hstack([cxr,ecg,labs,ctx]).astype(np.float32)
Xmeta_oof=meta_matrix(CXR_oof,ECG_oof,LABS_oof,CTX_tr)
Xmeta_vl =meta_matrix(CXR_vl ,ECG_vl ,LABS_vl ,CTX_vl)
Xmeta_te =meta_matrix(CXR_te ,ECG_te ,LABS_te ,CTX_te)
PROB_NAMES=[f"{m}:{l}" for m in MODALITIES for l in LABELS]
META_NAMES=PROB_NAMES+CTX_NAMES
print(f"Meta-features: {Xmeta_oof.shape[1]} (18 probas + {len(CTX_NAMES)} contexto)")
# Guardar las predicciones base para reproducibilidad
for nm,arr,df in [("oof_train",[CXR_oof,ECG_oof,LABS_oof],df_train),("val",[CXR_vl,ECG_vl,LABS_vl],df_val),("test",[CXR_te,ECG_te,LABS_te],df_test)]:
    cols={"hadm_id":df["hadm_id"].to_numpy()}
    for mi,mod in enumerate(MODALITIES):
        for j,l in enumerate(LABELS): cols[f"{mod}_{l.replace(' ','_')}"]=arr[mi][:,j]
    pd.DataFrame(cols).to_csv(OUT/f"base_preds_{nm}.csv",index=False)
print("Predicciones base guardadas (base_preds_*.csv).")


In [ ]:
# CELDA 9 · META-MODELOS (LogReg L2 + XGBoost) y BASELINES, evaluados sin leakage
def meta_ovr(make,Xtr,ytr,mtr,Xva,scale=False):
    P=np.full((len(Xva),N_LABELS),0.5,np.float32); sc=None
    if scale: sc=StandardScaler().fit(Xtr); Xtr=sc.transform(Xtr); Xva=sc.transform(Xva)
    for j in range(N_LABELS):
        sel=mtr[:,j]==1; Xj=Xtr[sel]; yj=ytr[sel,j]
        if len(np.unique(yj))<2: P[:,j]=float(yj.mean()) if len(yj) else 0.5; continue
        clf=make(); clf.fit(Xj,yj); P[:,j]=clf.predict_proba(Xva)[:,1]
    return P
def make_lr(C=0.5): return LogisticRegression(C=C,class_weight="balanced",max_iter=2000,solver="lbfgs")
def make_xgbmeta(): return xgb.XGBClassifier(n_estimators=60,max_depth=2,learning_rate=0.08,subsample=0.8,colsample_bytree=0.8,reg_lambda=5.0,tree_method="hist",eval_metric="logloss",n_jobs=4,random_state=SEED)

# Estimación honesta por K-fold sobre las OOF (el meta nunca ve, en cada fold, lo que predice)
def meta_cv(make,scale,Xmeta,k=5):
    kf=KFold(k,shuffle=True,random_state=SEED); P=np.zeros((len(Xmeta),N_LABELS),np.float32)
    for tr,va in kf.split(np.arange(len(Xmeta))):
        P[va]=meta_ovr(make,Xmeta[tr],y_train[tr],m_train[tr],Xmeta[va],scale=scale)
    return P
# Tuning ligero de C para LogReg sobre la CV interna
bestC,bestv=0.5,-1
for C in [0.05,0.1,0.3,0.5,1.0,2.0]:
    p=meta_cv(lambda C=C: make_lr(C),True,Xmeta_oof); v=multilabel_metrics(p,y_train,m_train)["macro_AP_path"]
    if v>bestv: bestv,bestC=v,C
print(f"Mejor C LogReg: {bestC} (CV macro={bestv:.4f})")
oof_LR =meta_cv(lambda: make_lr(bestC),True,Xmeta_oof)
oof_XGB=meta_cv(make_xgbmeta,False,Xmeta_oof)
# Baselines OOF: blending (promedio de modalidades) y mejor individual (CXR)
oof_BLEND=np.mean([CXR_oof,ECG_oof,LABS_oof],axis=0)
APPROACHES={"Stacking_LR":oof_LR,"Stacking_XGB":oof_XGB,"Blending":oof_BLEND,"CXR_solo":CXR_oof}
print("\nEstimación CV (sobre OOF):")
for k,p in APPROACHES.items():
    mm=multilabel_metrics(p,y_train,m_train); print(f"   {k:12s} macroP={mm['macro_AP_path']:.4f} core={mm['macro_AUC_core']:.4f}")



In [ ]:
# CELDA 10 · META FINAL + RECALIBRACIÓN + FUSIÓN vs MEJOR MONO-MODELO (evaluación en TEST)
# Esta es la celda que responde a la PREGUNTA CENTRAL del TFM: ¿aporta algo fusionar las tres
# modalidades frente a usar la mejor por separado? El EDA (§5, §8) predecía que sí, porque ninguna
# modalidad separa los diagnósticos en solitario. Aquí se comprueba CON INTERVALOS DE CONFIANZA.

val_LR  = meta_ovr(lambda: make_lr(bestC), Xmeta_oof, y_train, m_train, Xmeta_vl, scale=True)
test_LR = meta_ovr(lambda: make_lr(bestC), Xmeta_oof, y_train, m_train, Xmeta_te, scale=True)
val_XGB  = meta_ovr(make_xgbmeta, Xmeta_oof, y_train, m_train, Xmeta_vl, scale=False)
test_XGB = meta_ovr(make_xgbmeta, Xmeta_oof, y_train, m_train, Xmeta_te, scale=False)
val_BLEND = np.mean([CXR_vl, ECG_vl, LABS_vl], axis=0)
test_BLEND = np.mean([CXR_te, ECG_te, LABS_te], axis=0)

# Enfoques a comparar: 2 fusiones "inteligentes", 1 promedio simple y las 3 modalidades SOLAS.
TEST = {"Stacking_LR": (test_LR, val_LR), "Stacking_XGB": (test_XGB, val_XGB),
        "Blending": (test_BLEND, val_BLEND),
        "CXR_solo": (CXR_te, CXR_vl), "ECG_solo": (ECG_te, ECG_vl), "LABS_solo": (LABS_te, LABS_vl)}

# ══════════════════════════════════════════════════════════════════════════════
# RECALIBRACIÓN POST-FUSIÓN (§11 EDA: "recomprobar la calibración tras la fusión")
# ──────────────────────────────────────────────────────────────────────────────
# POR QUÉ: cada modalidad venía calibrada por separado, pero combinar probabilidades NO conserva la
#          calibración: un promedio de probabilidades calibradas está sistemáticamente SUB-confiado
#          (se comprime hacia el centro). Por eso se reajusta una isotónica sobre VAL para el enfoque
#          fusionado y se vuelve a medir el Brier.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc): si el Brier mejora mucho al recalibrar, es la
#          prueba de que la fusión desajusta las probabilidades y de que este paso es imprescindible
#          antes de mostrar cifras a un clínico.
# ══════════════════════════════════════════════════════════════════════════════
def recalibrar(val_p, test_p):
    cal = {}
    for j, l in enumerate(LABELS):
        s = m_val[:, j] == 1; yt = y_val[s, j]; yp = val_p[s, j]
        cal[l] = None if len(np.unique(yt)) < 2 else IsotonicRegression(out_of_bounds="clip").fit(yp, yt)
    def ap(P):
        O = P.copy()
        for j, l in enumerate(LABELS):
            if cal[l] is not None: O[:, j] = cal[l].predict(P[:, j])
        return O
    return ap(val_p), ap(test_p)

print("== METRICAS EN TEST · AP es la PRIMARIA (linea base = prevalencia) ==")
print("{:14s} {:>8} {:>16} {:>8}".format("Enfoque", "macroAP", "IC95% macroAP", "macroAUC"))
print("-" * 52)
RES = {}; rows = []
for k, (tp, vp) in TEST.items():
    vp_c, tp_c = recalibrar(vp, tp)                      # recalibrar SIEMPRE (tambien los mono, para comparar justo)
    thr = best_thresholds_by_f1(vp_c, y_val, m_val)
    mm = multilabel_metrics(tp_c, y_test, m_test, thresholds=thr)
    ci = bootstrap_ci_metric(tp_c, y_test, m_test, metric="ap")
    lo, hi = ci["macro_path"]
    RES[k] = {"m": mm, "ci": (lo, hi), "test_cal": tp_c, "val_cal": vp_c, "thr": thr}
    print("{:14s} {:8.4f} {:>16} {:8.4f}".format(k, mm["macro_AP_path"], f"[{lo:.3f},{hi:.3f}]", mm["macro_AUC_path"]))
    rows.append({"enfoque": k, "macro_AP": mm["macro_AP_path"], "AP_ci_lo": lo, "AP_ci_hi": hi,
                 "macro_AUC": mm["macro_AUC_path"], **{f"AP_{l}": mm[l]["AP"] for l in LABELS}})
pd.DataFrame(rows).to_csv(OUT / "comparativa_enfoques.csv", index=False)

# ══════════════════════════════════════════════════════════════════════════════
# ¿LA FUSIÓN SUPERA AL MEJOR MONO-MODELO? (criterio del EDA §11 y §5)
# ──────────────────────────────────────────────────────────────────────────────
# REGLA: si los IC bootstrap del mejor fusionado y del mejor mono SE SOLAPAN, la mejora NO es
# concluyente y hay que decirlo explicitamente. Con val=750 y test=464 la varianza es alta (§1).
# ══════════════════════════════════════════════════════════════════════════════
FUSION = ["Stacking_LR", "Stacking_XGB", "Blending"]
MONO = ["CXR_solo", "ECG_solo", "LABS_solo"]
best_fus = max(FUSION, key=lambda k: RES[k]["m"]["macro_AP_path"])
best_mono = max(MONO, key=lambda k: RES[k]["m"]["macro_AP_path"])
af, (lf, hf) = RES[best_fus]["m"]["macro_AP_path"], RES[best_fus]["ci"]
am, (lm, hm) = RES[best_mono]["m"]["macro_AP_path"], RES[best_mono]["ci"]
solapan = not (lf > hm or lm > hf)

print("\n" + "=" * 72)
print(f"MEJOR FUSION      : {best_fus:14s} macroAP={af:.4f}  IC95%=[{lf:.3f},{hf:.3f}]")
print(f"MEJOR MONO-MODELO : {best_mono:14s} macroAP={am:.4f}  IC95%=[{lm:.3f},{hm:.3f}]")
print(f"Diferencia        : {af-am:+.4f}")
print("=" * 72)
if solapan:
    print("VEREDICTO: los IC SE SOLAPAN -> la mejora de la fusion NO es concluyente.")
    print("           Con este tamano de test (464) no se puede afirmar que fusionar sea mejor.")
else:
    print("VEREDICTO: los IC NO se solapan -> la fusion supera al mejor mono-modelo de forma consistente.")
print("NOTA: la AP no es comparable entre etiquetas (su linea base es la prevalencia de cada una).")

# Brier antes/despues de recalibrar, para el mejor enfoque fusionado
b_pre, _ = calibration_report(TEST[best_fus][0], y_test, m_test)
b_post, curvas = calibration_report(RES[best_fus]["test_cal"], y_test, m_test)
cmp_b = b_pre.merge(b_post, on="etiqueta", suffixes=("_sin_recalibrar", "_recalibrado"))
cmp_b["mejora"] = cmp_b["Brier_sin_recalibrar"] - cmp_b["Brier_recalibrado"]
print(f"\n== Brier del mejor fusionado ({best_fus}) antes vs despues de RECALIBRAR ==")
print(cmp_b.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"   Mejora en {(cmp_b['mejora'] > 0).sum()}/{len(cmp_b)} etiquetas")
cmp_b.to_csv(OUT / "recalibracion_brier.csv", index=False)

# Puntos de operacion, consistencia y estratificacion del MEJOR enfoque global
best_all = max(RES, key=lambda k: RES[k]["m"]["macro_AP_path"])
PUNTOS = operating_points(RES[best_all]["val_cal"], y_val, m_val, sens_target=0.90, spec_target=0.90)
clin = pd.concat([clinical_report(RES[best_all]["test_cal"], y_test, m_test, PUNTOS[k], punto=nm)
                  for k, nm in [("f1", "f1"), ("cribado", "cribado_Se>=0.90"), ("confirm", "confirmacion_Sp>=0.90")]],
                 ignore_index=True)
clin.to_csv(OUT / "puntos_operacion_test.csv", index=False)
print(f"\n== PUNTOS DE OPERACION del mejor enfoque ({best_all}) ==")
cab = "    {:18s} {:>5} {:>6} {:>6} {:>6} {:>6} {:>4} {:>4}".format("Etiqueta", "thr", "Se", "Sp", "VPP", "VPN", "FN", "FP")
for punto in clin["punto"].unique():
    print("\n  · Punto " + str(punto) + ":"); print(cab)
    for _, r in clin[clin["punto"] == punto].iterrows():
        print("    {:18s} {:5.2f} {:6.3f} {:6.3f} {:6.3f} {:6.3f} {:4d} {:4d}".format(
            r["etiqueta"], r["umbral"], r["Se"], r["Sp"], r["VPP"], r["VPN"], int(r["FN"]), int(r["FP"])))

cons = consistency_no_finding(RES[best_all]["test_cal"])
print(f"\n== Consistencia: corr(P(Sin hallazgo), max P(patologia))={cons['corr_NoFinding_vs_maxPatologia']:.3f} "
      f"(debe ser NEGATIVA) · incoherentes={cons['pct_incoherentes']:.1f} %")
strat = stratified_report(df_test, RES[best_all]["test_cal"], y_test, m_test)
print("\n== Rendimiento por subgrupo (n<60 => posible ruido, no inequidad) ==")
print(strat.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
strat.to_csv(OUT / "estratificacion_subgrupos.csv", index=False)

json.dump({"mejor_global": best_all, "mejor_fusion": best_fus, "mejor_mono": best_mono,
           "macro_ap_fusion": af, "ic_fusion": [lf, hf],
           "macro_ap_mono": am, "ic_mono": [lm, hm],
           "diferencia": af - am, "ic_se_solapan": bool(solapan),
           "veredicto": ("mejora NO concluyente (IC solapados)" if solapan
                         else "la fusion supera al mejor mono-modelo"),
           "consistencia": cons,
           "per_label_mejor": {l: RES[best_all]["m"][l] for l in LABELS}},
          open(OUT / "summary_stacking_v1.json", "w", encoding="utf-8"), indent=2, default=str, ensure_ascii=False)
print("\nGuardados: comparativa_enfoques.csv · recalibracion_brier.csv · puntos_operacion_test.csv ·")
print("           estratificacion_subgrupos.csv · summary_stacking_v1.json")

# Alias de compatibilidad para las celdas de graficas posteriores
TESTM = {k: RES[k]["m"] for k in RES}   # antes se llamaba TESTM
BESTAP = best_all                          # mejor enfoque por AUC-PR
best_test = RES[best_all]["test_cal"]   # predicciones (calibradas) del mejor enfoque
best_thr = RES[best_all]["thr"]              # umbrales F1 del mejor enfoque



In [ ]:
# CELDA 11 · GRÁFICA COMPARATIVA (barras macro + por etiqueta)
fig,axes=plt.subplots(1,2,figsize=(15,5))
order=["CXR_solo","Blending","Stacking_LR","Stacking_XGB"]; cols=["#95a5a6","#f1c40f","#2980b9","#27ae60"]
ax=axes[0]; vals=[TESTM[k]["macro_AP_path"] for k in order]
ax.bar(order,vals,color=cols,alpha=0.85); ax.set_ylim(0.5,max(vals)+0.03); ax.set_ylabel("macro AUC (patologías, test)")
ax.set_title("Comparativa de enfoques de fusión")
for i,v in enumerate(vals): ax.text(i,v+0.003,f"{v:.4f}",ha="center",fontsize=9)
ax=axes[1]; x=np.arange(N_LABELS); wbar=0.2
for i,k in enumerate(order):
    ax.bar(x+(i-1.5)*wbar,[TESTM[k][l]["AUC"] for l in LABELS],wbar,label=k,color=cols[i],alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([l[:8] for l in LABELS],rotation=30,ha="right"); ax.axhline(0.5,color="gray",ls="--")
ax.set_ylabel("AUC (test)"); ax.set_title("AUC por etiqueta y enfoque"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG/"comparativa_enfoques.png",dpi=150,bbox_inches="tight"); plt.show()
print("Figura guardada.")



In [ ]:
# CELDA 12 · INTERPRETACIÓN: ¿qué modalidad pesa en cada enfermedad? (coeficientes LogReg)
# Se entrena un LogReg por etiqueta SOLO sobre las 18 probabilidades (sin contexto) para leer pesos limpios.
sc=StandardScaler().fit(Xmeta_oof[:, :18]); Xp=sc.transform(Xmeta_oof[:, :18])
W=np.zeros((N_LABELS,3))  # peso agregado por modalidad y etiqueta
coef_full=np.zeros((N_LABELS,18))
for j,l in enumerate(LABELS):
    sel=m_train[:,j]==1; yj=y_train[sel,j]
    if len(np.unique(yj))<2: continue
    clf=make_lr(bestC).fit(Xp[sel],yj); co=clf.coef_[0]; coef_full[j]=co
    for mi in range(3): W[j,mi]=co[mi*N_LABELS+j]   # peso de la modalidad mi para SU misma etiqueta j
fig,ax=plt.subplots(figsize=(7,5))
sns.heatmap(W,annot=True,fmt=".2f",cmap="RdBu_r",center=0,xticklabels=MODALITIES,yticklabels=LABELS,ax=ax,cbar_kws={"label":"peso LogReg"})
ax.set_title("Peso de cada modalidad por enfermedad (coef. del meta-modelo)")
plt.tight_layout(); plt.savefig(FIG/"pesos_modalidad_enfermedad.png",dpi=150,bbox_inches="tight"); plt.show()
print("Modalidad dominante por enfermedad (según |peso|):")
for j,l in enumerate(LABELS):
    mi=int(np.argmax(np.abs(W[j]))); print(f"   {l:18s} -> {MODALITIES[mi]} (pesos: "+", ".join(f'{MODALITIES[k]}={W[j,k]:+.2f}' for k in range(3))+")")


In [ ]:
# CELDA 13 · MATRICES DE CONFUSIÓN + ROC/PR del mejor enfoque (test)
fig,axes=plt.subplots(2,3,figsize=(13,8)); fig.suptitle(f"Mejor enfoque ({BESTAP}) — Matrices de confusión (test)",fontweight="bold")
for j,l in enumerate(LABELS):
    ax=axes[j//3,j%3]; s=m_test[:,j]==1; yt=y_test[s,j]; yp=(best_test[s,j]>=best_thr.get(l,0.5)).astype(int)
    if len(yt)==0: ax.axis("off"); continue
    cm=confusion_matrix(yt,yp,labels=[0,1]); sns.heatmap(cm,annot=True,fmt="d",cmap="BuGn",cbar=False,ax=ax,xticklabels=["P0","P1"],yticklabels=["R0","R1"]); ax.set_title(f"{l} (thr={best_thr.get(l,0.5):.2f})",fontsize=10)
plt.tight_layout(); plt.savefig(FIG/"confusion_mejor_enfoque.png",dpi=150,bbox_inches="tight"); plt.show()
fig,axes=plt.subplots(1,2,figsize=(14,5))
for j,l in enumerate(LABELS):
    s=m_test[:,j]==1; yt=y_test[s,j]; yp=best_test[s,j]
    if len(np.unique(yt))<2: continue
    fpr,tpr,_=roc_curve(yt,yp); axes[0].plot(fpr,tpr,label=f"{l} ({TESTM[BESTAP][l]['AUC']:.3f})")
    pr,rc,_=precision_recall_curve(yt,yp); axes[1].plot(rc,pr,label=f"{l} ({TESTM[BESTAP][l]['AP']:.3f})")
axes[0].plot([0,1],[0,1],"k--",alpha=0.4); axes[0].set_title("ROC (test)"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend(fontsize=8)
axes[1].set_title("Precisión-Recall (test)"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precisión"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG/"roc_pr_mejor_enfoque.png",dpi=150,bbox_inches="tight"); plt.show()
print("Figuras guardadas en", FIG)


---
## ✅ Conclusión — Stacking multimodal v1 (base)

Fusión tardía con **meta-modelo por etiqueta** (LogReg L2 y XGBoost) sobre 18 probabilidades + contexto
clínico, comparada contra el **promedio simple** y contra **cada modalidad en solitario**, con
**recalibración post-fusión** y **AUC-PR con IC bootstrap**.

### 📌 Para la documentación posterior (recuadros naranjas a redactar con los resultados)

- **LA PREGUNTA CENTRAL DEL TFM**: ¿supera la fusión al mejor mono-modelo? El notebook lo responde
  con IC. **Si los intervalos se solapan, la mejora NO es concluyente** y hay que escribirlo tal cual.
  Con un test de 464 pacientes, una diferencia de pocas milésimas no demuestra nada.
- **Contexto de lo ya medido**: LABS v1 = 0,3968 [0,368–0,436] · LABS v2 = 0,3910 [0,362–0,429] ·
  ECG v1 = 0,3667 [0,339–0,405]. Ya se vio que **v2 no supera a v1 en tabular**: el techo de una
  modalidad se alcanza pronto. La fusión tiene que batir a la **mejor** de las tres, no a la media.
- **Aportación diferencial por patología** (§5 EDA): se espera que la imagen lidere **Atelectasia y
  Opacidad** (las analíticas rondan el azar ahí) y que el tabular aporte en **Edema, Derrame y
  Cardiomegalia**. Si la fusión gana, debería ganar sobre todo donde las modalidades se complementan.
- **Recalibración**: comparar el Brier antes y después. Una mejora grande demuestra que fusionar
  desajusta las probabilidades y que este paso es imprescindible antes de enseñar cifras a un clínico.
- **Puntos de operación**: en los módulos individuales, ni el cribado ni la confirmación resultaban
  clínicamente asumibles. **La prueba de fuego de la fusión es si alguno de los dos puntos pasa a
  serlo.** Si tampoco, la conclusión honesta es que el sistema sirve como apoyo, no como decisor.
- **Interpretación por modalidad** (celda 12): qué peso da el meta a cada modalidad en cada
  enfermedad. Es lo que conecta el resultado con la fisiología y da valor clínico a la memoria.
- **Consistencia**: la correlación entre P(*Sin hallazgo*) y max P(patología) debe ser **negativa**.
- **Equidad**: subgrupos con n<60 ⇒ ruido, no inequidad. No concluir sin IC.
- ⚠️ Las cifras de fusión **anteriores no son comparables**: aquel código evaluaba contra otro
  etiquetado (el −1 como negativo, sin enmascarar) y con AUC-ROC como métrica.

